# Prompt-CAM end-to-end runner

This notebook automates the documented Prompt-CAM flow:
1. clone/update repo
2. create conda env + install dependencies
3. prepare data/checkpoints folders
4. run demo notebook, training, and visualization commands

> **Note:** Commands are written to follow the official README from `Imageomics/Prompt_CAM` and may require GPU, large datasets, and checkpoint downloads.

In [ ]:
from pathlib import Path
import os
import shlex
import subprocess

WORKSPACE = Path.cwd()
REPO_URL = "https://github.com/Imageomics/Prompt_CAM.git"
REPO_DIR = WORKSPACE / "Prompt_CAM"
ENV_NAME = "prompt_cam"

# Editable runtime settings
DATA_PATH = REPO_DIR / "data" / "images"
MODEL = "dino"  # dino | dinov2
DATASET = "cub" # cub | pet | dog | car | birds_525
VIS_CLS = 23
GPU_DEVICES = "0"
TRAIN_GPUS = "0,1,2,3"
NPROC_PER_NODE = 4

# Optional: set to a local model checkpoint once downloaded
CHECKPOINT_PATH = REPO_DIR / "checkpoints" / MODEL / DATASET / "model.pt"
CONFIG_PATH = REPO_DIR / "experiment" / "config" / "prompt_cam" / MODEL / DATASET / "args.yaml"

In [ ]:
def run(cmd, cwd=None, env=None, check=True):
    print(f"\n$ {cmd}")
    completed = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=env,
        shell=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(completed.stdout)
    if check and completed.returncode != 0:
        raise RuntimeError(f"Command failed ({completed.returncode}): {cmd}")
    return completed

## 1) Clone (or update) Prompt_CAM

In [ ]:
if not REPO_DIR.exists():
    run(f"git clone {shlex.quote(REPO_URL)} {shlex.quote(str(REPO_DIR))}")
else:
    run("git pull", cwd=REPO_DIR)

## 2) Environment setup

Equivalent to README:
```bash
conda create -n prompt_cam python=3.10
conda activate prompt_cam
source env_setup.sh
```

In [ ]:
run(f"conda create -y -n {ENV_NAME} python=3.10")
run(f"bash -lc 'source $(conda info --base)/etc/profile.d/conda.sh && conda activate {ENV_NAME} && source env_setup.sh'", cwd=REPO_DIR)

## 3) Ensure expected folder layout exists

In [ ]:
(REPO_DIR / "checkpoints" / MODEL / DATASET).mkdir(parents=True, exist_ok=True)
(REPO_DIR / "pretrained_weights").mkdir(parents=True, exist_ok=True)
(REPO_DIR / "visualization" / MODEL / DATASET).mkdir(parents=True, exist_ok=True)
print("Created/verified core folders.")
print(f"DATA_PATH expected under: {DATA_PATH}")

## 4) Run the demo notebook non-interactively

This executes all cells in `demo.ipynb` and writes an executed copy.

In [ ]:
run(
    "bash -lc 'source $(conda info --base)/etc/profile.d/conda.sh && "
    f"conda activate {ENV_NAME} && "
    "jupyter nbconvert --to notebook --execute demo.ipynb --output demo.executed.ipynb'",
    cwd=REPO_DIR,
)

## 5) Training command (README multi-GPU example)

Adjust `TRAIN_GPUS`, `NPROC_PER_NODE`, `MODEL`, and `DATASET` in the config cell.

In [ ]:
train_cmd = (
    "bash -lc 'source $(conda info --base)/etc/profile.d/conda.sh && "
    f"conda activate {ENV_NAME} && "
    f"CUDA_VISIBLE_DEVICES={TRAIN_GPUS} torchrun --nproc_per_node={NPROC_PER_NODE} "
    f"main.py --config {CONFIG_PATH} --gpu_num {NPROC_PER_NODE}'"
)
print(train_cmd)
# Uncomment the next line to run training:
# run(train_cmd, cwd=REPO_DIR)

## 6) Visualization / evaluation command

In [ ]:
vis_cmd = (
    "bash -lc 'source $(conda info --base)/etc/profile.d/conda.sh && "
    f"conda activate {ENV_NAME} && "
    f"CUDA_VISIBLE_DEVICES={GPU_DEVICES} python visualize.py "
    f"--config {CONFIG_PATH} "
    f"--checkpoint {CHECKPOINT_PATH} "
    f"--vis_cls {VIS_CLS}'"
)
print(vis_cmd)
# Uncomment the next line to run visualization:
# run(vis_cmd, cwd=REPO_DIR)

## 7) (Optional) Sweep multiple configs

Use this to run training+visualization for several model/dataset combinations.

In [ ]:
configs = [
    ("dino", "cub", 23),
    ("dino", "pet", 1),
    ("dinov2", "pet", 1),
]

for model, dataset, vis_cls in configs:
    cfg = REPO_DIR / "experiment" / "config" / "prompt_cam" / model / dataset / "args.yaml"
    ckpt = REPO_DIR / "checkpoints" / model / dataset / "model.pt"
    print({"model": model, "dataset": dataset, "config": str(cfg), "checkpoint": str(ckpt), "vis_cls": vis_cls})